In [1]:
import jax 
import jax.numpy as jnp
import jax.random as jrandom
import jax.scipy.stats as stats

In [2]:
from probjax.core.custom_primitives.random_variable import rv_p

In [10]:
sampling_to_log_prob = {
    '_normal': lambda x: stats.norm.logpdf(x).sum(),
    '_uniform': lambda x,a,b: jax.scipy.stats.uniform.logpdf(x, a, (b-a)).sum(),
    '_bernoulli':  lambda x, p: jax.scipy.stats.bernoulli.logpmf(x,p).sum(),
    '_binomial': lambda x, n, p: jax.scipy.stats.binom.logpmf(x, n, p).sum(),
    '_gamma': lambda x, a: jax.scipy.stats.gamma.logpdf(x, a).sum(),
    '_exponential': lambda x: jax.scipy.stats.expon.logpdf(x).sum(),
    '_poisson': lambda x, a: jax.scipy.stats.poisson.logpmf(x, a).sum(),
    '_geometric': lambda x, p: jax.scipy.stats.geom.logpmf(x, p).sum(),
    '_cauchy': lambda x, a, b: jax.scipy.stats.cauchy.logpdf(x, a, b).sum(),
    '_pareto': lambda x, b: jax.scipy.stats.pareto.logpdf(x, b).sum(),
    '_chisquare': lambda x, df: jax.scipy.stats.chi2.logpdf(x, df).sum(),
    '_dirichlet': lambda x, a: jax.scipy.stats.dirichlet.logpdf(x, a).sum(),
    '_truncated_normal': lambda x, a, b: jax.scipy.stats.truncnorm.logpdf(x, a, b).sum(),
    '_multivariate_normal' : lambda x, mean, cov: jax.scipy.stats.multivariate_normal.logpdf(x, mean, cov).sum(),
    '_laplace': lambda x: jax.scipy.stats.laplace.logpdf(x).sum(),
    '_logistic': lambda x: jax.scipy.stats.logistic.logpdf(x).sum(),
    '_gumbel': lambda x:  NotImplementedError,
    '_maxwell': lambda x:  NotImplementedError,
    '_double_sided_maxwell': lambda x, loc,scale:  NotImplementedError,
    '_f': lambda x, df1, df2:  NotImplementedError,
    '_t': lambda x, df:  NotImplementedError,
    '_rademacher': lambda x:  NotImplementedError,
    '_wald': lambda x, a:  NotImplementedError,
}

In [11]:
def g(key):
    x1 = jax.random.normal(key)
    x2 = jax.random.uniform(key)
    x3 = jax.random.bernoulli(key)
    #x4 = jax.random.gamma(key, 2.)
    #y = x1 + x3 #+ x2 #+ x3 + x4
    return x3

In [12]:
jaxpr = jax.make_jaxpr(g)(jrandom.PRNGKey(0))

In [13]:
from jax.core import JaxprEqn

def update_eqn(eqn, name):
    sampling_fn_jaxpr = eqn.params["jaxpr"]
    sampling_name = eqn.params["name"]
    
    #print(sampling_fn_jaxpr.jaxpr.constvars)
    try:
        log_prob_fn = sampling_to_log_prob[sampling_name]
    except KeyError:
        raise NotImplementedError(f"Sampling function {sampling_name} no log_prob implemented")
    invals = jax._src.core.safe_map(lambda x: x.aval, eqn.outvars)
    additional_invals = jax._src.core.safe_map(lambda x: x.aval, sampling_fn_jaxpr.jaxpr.invars[1:])
    print(invals, additional_invals)
    log_prob_fn_jaxpr = jax.make_jaxpr(log_prob_fn)(*invals, *additional_invals)
    # print(log_prob_fn_jaxpr)
    
    params = {"sampling_fn_jaxpr": sampling_fn_jaxpr, "log_prob_fn_jaxpr": log_prob_fn_jaxpr, "name": name}
    new_eqn = JaxprEqn(eqn.invars, eqn.outvars, rv_p, params, eqn.effects, eqn.source_info)
    return new_eqn

def trace_rv(jaxpr):
    for i in range(len(jaxpr.eqns)):
        eqn = jaxpr.eqns[i]
        if eqn.primitive is not rv_p and "name" in eqn.params:
            new_eqn = update_eqn(eqn, "x" + str(i))
            jaxpr.eqns[i] = new_eqn
    return jaxpr

In [14]:
from probjax.core import joint_sample, log_potential_fn

In [15]:
jaxpr = jax.make_jaxpr(g)(jrandom.PRNGKey(0))

In [25]:
jaxpr

let x3 = { lambda ; a:key<fry>[] b:f32[] c:f32[]. let
    d:f32[] = convert_element_type[new_dtype=float32 weak_type=False] b
    e:f32[] = convert_element_type[new_dtype=float32 weak_type=False] c
    f:u32[] = random_bits[bit_width=32 shape=()] a
    g:u32[] = shift_right_logical f 9
    h:u32[] = or g 1065353216
    i:f32[] = bitcast_convert_type[new_dtype=float32] h
    j:f32[] = sub i 1.0
    k:f32[] = sub e d
    l:f32[] = mul j k
    m:f32[] = add l d
    n:f32[] = reshape[dimensions=None new_sizes=()] m
    o:f32[] = max d n
  in (o,) } in
let _where = { lambda ; p:bool[] q:f32[] r:f32[]. let
    s:f32[] = convert_element_type[new_dtype=float32 weak_type=False] q
    t:f32[] = select_n p r s
  in (t,) } in
let _where1 = { lambda ; u:bool[] v:f32[] w:f32[]. let
    x:f32[] = select_n u w v
  in (x,) } in
{ lambda ; y:u32[2]. let
    z:key<fry>[] = random_wrap[impl=fry] y
    _:f32[] = random_variable[
      log_prob_fn_jaxpr={ lambda ; ba:f32[]. let
          bb:f32[] = integer_

In [16]:
new_jaxpr = trace_rv(jaxpr)

[ShapedArray(float32[])] []
[ShapedArray(float32[])] [ShapedArray(float32[], weak_type=True), ShapedArray(float32[], weak_type=True)]
[ShapedArray(bool[])] [ShapedArray(float32[])]


In [17]:
f_traced = jax.core.jaxpr_as_fun(new_jaxpr)

In [18]:
f_traced(jrandom.PRNGKey(0))

[Array(True, dtype=bool)]

In [19]:
from probjax.core import joint_sample, log_potential_fn

In [20]:
sampler = joint_sample(f_traced)

samples = sampler(jrandom.PRNGKey(0))

In [21]:
samples

{'x1': Array(-0.206, dtype=float32),
 'x3': Array(0.418, dtype=float32),
 'x5': Array(True, dtype=bool)}

In [23]:
log_potential_fn(f_traced)(**samples)

Array(-1.633, dtype=float32)

In [24]:
jaxpr = jax.make_jaxpr(log_potential_fn(f_traced))(x1=1.)

KeyError: 'x3'